# Enhanced Models with Regression-to-Classification Approach

This notebook implements:
1. S&P 500 data from 1990 with proper time handling
2. Enhanced feature engineering
3. Regression models (BiLSTM, BiGRU, LightGBM) for price prediction
4. Convert regression predictions to directional classification
5. Performance evaluation for both regression and classification

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, classification_report, confusion_matrix
import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import pytz

## Data Collection and Preprocessing

In [ ]:
def fetch_market_data(start_date='1990-01-01', end_date='2024-12-31'):
    """Fetch S&P 500 data with proper timezone handling"""
    df = yf.download('^GSPC', start=start_date, end=end_date)
    df.index = df.index.tz_localize(pytz.UTC)
    return df

def add_enhanced_technical_indicators(df):
    """Add comprehensive technical indicators"""
    # Price-based indicators
    df['Returns'] = df['Close'].pct_change()
    df['High-Low'] = df['High'] - df['Low']
    df['Open-Close'] = df['Open'] - df['Close']
    
    # Multiple timeframe moving averages
    for window in [5, 10, 20, 50, 200]:
        df[f'MA{window}'] = df['Close'].rolling(window=window).mean()
        df[f'MA{window}_Slope'] = df[f'MA{window}'].diff()
    
    # Volatility indicators
    df['Daily_Volatility'] = df['Returns'].rolling(window=20).std()
    
    # Bollinger Bands
    rolling_std = df['Close'].rolling(window=20).std()
    df['BB_Upper'] = df['Close'].rolling(window=20).mean() + (2 * rolling_std)
    df['BB_Lower'] = df['Close'].rolling(window=20).mean() - (2 * rolling_std)
    df['BB_Width'] = (df['BB_Upper'] - df['BB_Lower']) / df['Close']
    
    # Volume indicators
    df['Volume_MA5'] = df['Volume'].rolling(window=5).mean()
    df['Volume_MA20'] = df['Volume'].rolling(window=20).mean()
    df['Volume_Ratio'] = df['Volume'] / df['Volume_MA20']
    
    return df

def prepare_features(df):
    """Prepare features and regression target"""
    df = add_enhanced_technical_indicators(df)
    
    # Next day's return as regression target
    df['Target_Return'] = df['Close'].shift(-1) - df['Close']
    
    # Store actual direction for later evaluation
    df['Direction'] = 0
    df.loc[df['Target_Return'] > 0, 'Direction'] = 1
    df.loc[df['Target_Return'] < 0, 'Direction'] = -1
    
    return df.dropna()

# Fetch and prepare data
df = fetch_market_data()
df = prepare_features(df)
print(f"Data shape: {df.shape}")
print(f"Features created: {list(df.columns)}")

## Temporal Data Split

In [ ]:
def temporal_split(df):
    """Split data temporally"""
    train_end = '2015-12-31'
    val_end = '2020-12-31'
    
    train_data = df[df.index <= train_end]
    val_data = df[(df.index > train_end) & (df.index <= val_end)]
    test_data = df[df.index > val_end]
    
    print(f"Train period: {train_data.index.min()} to {train_data.index.max()}")
    print(f"Validation period: {val_data.index.min()} to {val_data.index.max()}")
    print(f"Test period: {test_data.index.min()} to {test_data.index.max()}")
    
    return train_data, val_data, test_data

train_data, val_data, test_data = temporal_split(df)

## Data Preparation

In [ ]:
def prepare_sequences(data, seq_length=20):
    """Prepare sequences for deep learning models"""
    feature_cols = [col for col in data.columns if col not in ['Target_Return', 'Direction']]
    
    sequences = []
    returns = []
    directions = []
    
    for i in range(len(data) - seq_length):
        seq = data[feature_cols].iloc[i:(i + seq_length)].values
        ret = data['Target_Return'].iloc[i + seq_length]
        direction = data['Direction'].iloc[i + seq_length]
        
        sequences.append(seq)
        returns.append(ret)
        directions.append(direction)
    
    return np.array(sequences), np.array(returns), np.array(directions)

class StockDataset(Dataset):
    def __init__(self, sequences, returns, directions):
        self.sequences = torch.FloatTensor(sequences)
        self.returns = torch.FloatTensor(returns)
        self.directions = torch.LongTensor(directions)
        
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.returns[idx], self.directions[idx]

# Prepare sequences
X_train, y_train_reg, y_train_cls = prepare_sequences(train_data)
X_val, y_val_reg, y_val_cls = prepare_sequences(val_data)
X_test, y_test_reg, y_test_cls = prepare_sequences(test_data)

# Create datasets
train_dataset = StockDataset(X_train, y_train_reg, y_train_cls)
val_dataset = StockDataset(X_val, y_val_reg, y_val_cls)
test_dataset = StockDataset(X_test, y_test_reg, y_test_cls)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

## Model Architectures

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=2, dropout=0.2):
        super(BiLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                           batch_first=True, bidirectional=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        lstm_out = lstm_out[:, -1, :]
        output = self.fc(lstm_out)
        return output.squeeze()
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

class BiGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=2, dropout=0.2):
        super(BiGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers,
                          batch_first=True, bidirectional=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        
    def forward(self, x):
        gru_out, _ = self.gru(x)
        gru_out = gru_out[:, -1, :]
        output = self.fc(gru_out)
        return output.squeeze()
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Initialize models
input_dim = X_train.shape[2]
hidden_dim = 128

bilstm_model = BiLSTM(input_dim, hidden_dim)
bigru_model = BiGRU(input_dim, hidden_dim)

print(f"BiLSTM parameters: {bilstm_model.count_parameters():,}")
print(f"BiGRU parameters: {bigru_model.count_parameters():,}")

## Model Training

In [ ]:
def train_deep_model(model, train_loader, val_loader, device, model_name):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    best_val_loss = float('inf')
    patience = 10
    patience_counter = 0
    
    train_losses = []
    val_losses = []
    
    for epoch in range(100):
        model.train()
        train_loss = 0
        for sequences, returns, _ in train_loader:
            sequences, returns = sequences.to(device), returns.to(device)
            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, returns)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for sequences, returns, _ in val_loader:
                sequences, returns = sequences.to(device), returns.to(device)
                outputs = model(sequences)
                val_loss += criterion(outputs, returns).item()
        
        train_losses.append(train_loss/len(train_loader))
        val_losses.append(val_loss/len(val_loader))
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch}')
                break
                
        if epoch % 10 == 0:
            print(f'{model_name} - Epoch {epoch}: '
                  f'Train Loss = {train_loss/len(train_loader):.4f}, '
                  f'Val Loss = {val_loss/len(val_loader):.4f}')
    
    return model, train_losses, val_losses

# Train models
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bilstm_model, bilstm_train_losses, bilstm_val_losses = train_deep_model(
    bilstm_model, train_loader, val_loader, device, "BiLSTM")
bigru_model, bigru_train_losses, bigru_val_losses = train_deep_model(
    bigru_model, train_loader, val_loader, device, "BiGRU")

# Train LightGBM
lgb_train = lgb.Dataset(X_train.reshape(X_train.shape[0], -1), label=y_train_reg)
lgb_val = lgb.Dataset(X_val.reshape(X_val.shape[0], -1), label=y_val_reg)

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9
}

lgb_model = lgb.train(lgb_params,
                      lgb_train,
                      valid_sets=[lgb_train, lgb_val],
                      num_boost_round=1000,
                      early_stopping_rounds=50,
                      verbose_eval=100)

## Model Evaluation

In [ ]:
def evaluate_model(model, test_loader, device, model_type='deep'):
    if model_type == 'deep':
        model.eval()
        predictions = []
        actuals_reg = []
        actuals_cls = []
        
        with torch.no_grad():
            for sequences, returns, directions in test_loader:
                sequences = sequences.to(device)
                outputs = model(sequences)
                predictions.extend(outputs.cpu().numpy())
                actuals_reg.extend(returns.numpy())
                actuals_cls.extend(directions.numpy())
                
        predictions = np.array(predictions)
        actuals_reg = np.array(actuals_reg)
        actuals_cls = np.array(actuals_cls)
        
    else:  # LightGBM
        predictions = model.predict(X_test.reshape(X_test.shape[0], -1))
        actuals_reg = y_test_reg
        actuals_cls = y_test_cls
    
    # Convert regression predictions to directional predictions
    pred_directions = np.zeros_like(predictions)
    pred_directions[predictions > 0] = 1
    pred_directions[predictions < 0] = -1
    
    return {
        'rmse': np.sqrt(mean_squared_error(actuals_reg, predictions)),
        'classification_report': classification_report(actuals_cls, pred_directions),
        'confusion_matrix': confusion_matrix(actuals_cls, pred_directions),
        'predictions': predictions,
        'actuals_reg': actuals_reg,
        'actuals_cls': actuals_cls,
        'pred_directions': pred_directions
    }

# Evaluate models
bilstm_results = evaluate_model(bilstm_model, test_loader, device, 'deep')
bigru_results = evaluate_model(bigru_model, test_loader, device, 'deep')
lgb_results = evaluate_model(lgb_model, test_loader, device, 'lgb')

# Print results
for name, results in [('BiLSTM', bilstm_results), 
                     ('BiGRU', bigru_results), 
                     ('LightGBM', lgb_results)]:
    print(f"\n{name} Results:")
    print(f"RMSE: {results['rmse']:.4f}")
    print("\nClassification Report:")
    print(results['classification_report'])

## Visualization

In [ ]:
def plot_confusion_matrices(results_dict):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for ax, (name, results) in zip(axes, results_dict.items()):
        sns.heatmap(results['confusion_matrix'], 
                    annot=True, fmt='d', ax=ax)
        ax.set_title(f'{name} Confusion Matrix')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
    
    plt.tight_layout()
    plt.show()

def plot_training_curves():
    plt.figure(figsize=(12, 5))
    
    plt.plot(bilstm_train_losses, label='BiLSTM Train')
    plt.plot(bilstm_val_losses, label='BiLSTM Val')
    plt.plot(bigru_train_losses, label='BiGRU Train')
    plt.plot(bigru_val_losses, label='BiGRU Val')
    
    plt.title('Training Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

# Plot results
results_dict = {
    'BiLSTM': bilstm_results,
    'BiGRU': bigru_results,
    'LightGBM': lgb_results
}

plot_confusion_matrices(results_dict)
plot_training_curves()